In [1]:
%pip install --upgrade --quiet google-genai nest-asyncio==1.5.9

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 9.0 MB/s eta 0:00:00


In [1]:
import pandas as pd
from inspect import cleandoc
from IPython.display import display, Markdown

import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
from vertexai.evaluation import (
    MetricPromptTemplateExamples,
    EvalTask,
    PairwiseMetric,
    PairwiseMetricPromptTemplate,
    PointwiseMetric,
    PointwiseMetricPromptTemplate,
)

pd.set_option("display.max_colwidth", None)


In [2]:
vertexai.init(project="qwiklabs-gcp-00-30ed074629af", location="us-central1")


In [3]:
from inspect import cleandoc

hourly_rates = cleandoc("""
  Screenwriter: $40
  Actor: $25
  Director: $30
  Camera Operator: $35
  Sound Engineer: $20
  Editor: $30
  """)

planning_notes = cleandoc("""
 Phases of Production:
   Writing:
   The Screenwriter will write the script.
   They need 72 hours to do so.


   Pre-Production:
   The Director needs time to analyze the script.
   They will work on it for 36 hours.
   The Camera Operator will join the director for 24 hours of planning.


   Production Phase 1
   The first three days of filming will require the director, 4 actors, the camera operator, and the sound engineer


   Production Phase 2
   The next three days of filming will require the director, 8 actors, the camera operator, and the sound engineer


   Post-Production
   The editor will take 64 hours to edit the film.
   The director will work with the editor for 24 hours during this phase.
""")


In [4]:
tasks = [
    """What is the cost of each phase of production?
    If days are mentioned, assume an 8 hour work day.""",

    """How many days will each phase require? Assume an
    8 hour work day. If multiple people are working in parallel,
    do not add those times together, but only use the longest time.
    Also include a count of the total number of days of the entire
    project.""",

    """Prepare a text schedule for all phases of the film starting
    on Feb 3, 2025. The whole crew should be off Saturdays
    and Sundays."""
]


In [5]:
prompt_template = cleandoc("""
  <instructions>
  Prepare a document to fulfill the task based on the context provided.
  </instructions>
<task>
  {task}
  </task>
<context>
  {context}
  </context>
""")


In [6]:
from IPython.display import Markdown, display
from vertexai.generative_models import GenerativeModel, GenerationConfig

# Create the two model objects
llm_pro = GenerativeModel(
    "gemini-2.5-pro",
    generation_config=GenerationConfig(temperature=0)
)

llm_flash = GenerativeModel(
    "gemini-2.0-flash-001",
    generation_config=GenerationConfig(temperature=0)
)


In [7]:
context = hourly_rates + "\n\n" + planning_notes

# Second task (index 1) prompt
prompt_task2 = prompt_template.format(
    task=tasks[1],
    context=context
)


In [8]:
# Pro
resp_pro = llm_pro.generate_content(prompt_task2)
text_pro = resp_pro.text if hasattr(resp_pro, "text") else str(resp_pro)

# Flash
resp_flash = llm_flash.generate_content(prompt_task2)
text_flash = resp_flash.text if hasattr(resp_flash, "text") else str(resp_flash)

# Render nicely (Gemini outputs often include Markdown)
display(Markdown("## Gemini 2.5 Pro — Task 2 Response"))
display(Markdown(text_pro))

display(Markdown("## Gemini 2.0 Flash — Task 2 Response"))
display(Markdown(text_flash))


## Gemini 2.5 Pro — Task 2 Response

Based on the context provided, here is a breakdown of the days required for each phase and the total project duration, assuming an 8-hour workday.

### **Phase Durations**

*   **Writing:**
    *   The Screenwriter requires 72 hours.
    *   Calculation: 72 hours / 8 hours per day = **9 days**

*   **Pre-Production:**
    *   The Director works for 36 hours, and the Camera Operator works for 24 hours in parallel. The longest duration is used.
    *   Longest duration: 36 hours.
    *   Calculation: 36 hours / 8 hours per day = **4.5 days**

*   **Production Phase 1:**
    *   The duration is explicitly stated in the context.
    *   Total: **3 days**

*   **Production Phase 2:**
    *   The duration is explicitly stated in the context.
    *   Total: **3 days**

*   **Post-Production:**
    *   The Editor works for 64 hours, and the Director works for 24 hours in parallel. The longest duration is used.
    *   Longest duration: 64 hours.
    *   Calculation: 64 hours / 8 hours per day = **8 days**

---

### **Total Project Duration**

*   Writing: 9 days
*   Pre-Production: 4.5 days
*   Production Phase 1: 3 days
*   Production Phase 2: 3 days
*   Post-Production: 8 days
*   **Total: 27.5 days**

## Gemini 2.0 Flash — Task 2 Response

Here's a breakdown of the project timeline, calculated in days based on an 8-hour workday:

**Phase Breakdown:**

*   **Writing:**
    *   Screenwriter: 72 hours / 8 hours/day = 9 days

*   **Pre-Production:**
    *   Director: 36 hours / 8 hours/day = 4.5 days
    *   Camera Operator: 24 hours / 8 hours/day = 3 days
    *   *Longest Time:* 4.5 days

*   **Production Phase 1:**
    *   3 days (given)

*   **Production Phase 2:**
    *   3 days (given)

*   **Post-Production:**
    *   Editor: 64 hours / 8 hours/day = 8 days
    *   Director: 24 hours / 8 hours/day = 3 days
    *   *Longest Time:* 8 days

**Total Project Days:**

9 + 4.5 + 3 + 3 + 8 = **27.5 days**

In [9]:
# Helper math to sanity-check days
HOURS_PER_DAY = 8.0

days = {}

# Writing
days["Writing"] = 72 / HOURS_PER_DAY  # screenwriter only

# Pre-Production (director 36h vs camera 24h in parallel → take longest = 36h)
days["Pre-Production"] = 36 / HOURS_PER_DAY

# Production phases are given in days directly (3 days each)
days["Production Phase 1"] = 3
days["Production Phase 2"] = 3

# Post-Production (editor 64h with director 24h overlapping → take longest = 64h)
days["Post-Production"] = 64 / HOURS_PER_DAY

total_days = sum(days.values())

import pandas as pd
pd.DataFrame(
    [(k, round(v, 2)) for k, v in days.items()] + [("TOTAL", round(total_days, 2))],
    columns=["Phase", "Days (8h/day, parallel=max)"]
)


,Phase,"Days (8h/day, parallel=max)"
0,Writing,9.0
1,Pre-Production,4.5
2,Production Phase 1,3.0
3,Production Phase 2,3.0
4,Post-Production,8.0
5,TOTAL,27.5


In [10]:
for k, v in days.items():
    print(f"{k}: {v:.2f} days")
print(f"TOTAL: {total_days:.2f} days")


Writing: 9.00 days
Pre-Production: 4.50 days
Production Phase 1: 3.00 days
Production Phase 2: 3.00 days
Post-Production: 8.00 days
TOTAL: 27.50 days


In [11]:
from IPython.display import Markdown, display

def make_prompt(task_text: str) -> str:
    return prompt_template.format(task=task_text, context=context)

prompts = [make_prompt(t) for t in tasks]

pro_texts = []
flash_texts = []
for p in prompts:
    pro_resp   = llm_pro.generate_content(p)
    flash_resp = llm_flash.generate_content(p)

    pro_texts.append(getattr(pro_resp, "text", str(pro_resp)))
    flash_texts.append(getattr(flash_resp, "text", str(flash_resp)))

# (Optional) quick peek
display(Markdown("### Sample — Gemini Pro (baseline) output for Task 1"))
display(Markdown(pro_texts[0]))
display(Markdown("### Sample — Gemini Flash (candidate) output for Task 1"))
display(Markdown(flash_texts[0]))


### Sample — Gemini Pro (baseline) output for Task 1

Based on the information provided, here is the cost breakdown for each phase of production.

### **Writing**
*   **Screenwriter:** 72 hours @ $40/hour = $2,880
*   **Total Phase Cost: $2,880**

---

### **Pre-Production**
*   **Director:** 36 hours @ $30/hour = $1,080
*   **Camera Operator:** 24 hours @ $35/hour = $840
*   **Total Phase Cost: $1,920**

---

### **Production Phase 1**
This phase lasts for 3 days, which is a total of 24 work hours (3 days x 8 hours/day).
*   **Director:** 24 hours @ $30/hour = $720
*   **4 Actors:** 24 hours x 4 actors @ $25/hour = $2,400
*   **Camera Operator:** 24 hours @ $35/hour = $840
*   **Sound Engineer:** 24 hours @ $20/hour = $480
*   **Total Phase Cost: $4,440**

---

### **Production Phase 2**
This phase lasts for 3 days, which is a total of 24 work hours (3 days x 8 hours/day).
*   **Director:** 24 hours @ $30/hour = $720
*   **8 Actors:** 24 hours x 8 actors @ $25/hour = $4,800
*   **Camera Operator:** 24 hours @ $35/hour = $840
*   **Sound Engineer:** 24 hours @ $20/hour = $480
*   **Total Phase Cost: $6,840**

---

### **Post-Production**
*   **Editor:** 64 hours @ $30/hour = $1,920
*   **Director:** 24 hours @ $30/hour = $720
*   **Total Phase Cost: $2,640**

### Sample — Gemini Flash (candidate) output for Task 1

Here's a breakdown of the cost for each phase of production:

**Writing:**

*   Screenwriter: 72 hours * $40/hour = $2880

**Pre-Production:**

*   Director: 36 hours * $30/hour = $1080
*   Camera Operator: 24 hours * $35/hour = $840
*   **Total Pre-Production Cost:** $1080 + $840 = $1920

**Production Phase 1 (3 days):**

*   Director: 3 days * 8 hours/day * $30/hour = $720
*   Actors (4): 3 days * 8 hours/day * $25/hour * 4 actors = $2400
*   Camera Operator: 3 days * 8 hours/day * $35/hour = $840
*   Sound Engineer: 3 days * 8 hours/day * $20/hour = $480
*   **Total Production Phase 1 Cost:** $720 + $2400 + $840 + $480 = $4440

**Production Phase 2 (3 days):**

*   Director: 3 days * 8 hours/day * $30/hour = $720
*   Actors (8): 3 days * 8 hours/day * $25/hour * 8 actors = $4800
*   Camera Operator: 3 days * 8 hours/day * $35/hour = $840
*   Sound Engineer: 3 days * 8 hours/day * $20/hour = $480
*   **Total Production Phase 2 Cost:** $720 + $4800 + $840 + $480 = $6840

**Post-Production:**

*   Editor: 64 hours * $30/hour = $1920
*   Director: 24 hours * $30/hour = $720
*   **Total Post-Production Cost:** $1920 + $720 = $2640

**Summary of Costs:**

*   **Writing:** $2880
*   **Pre-Production:** $1920
*   **Production Phase 1:** $4440
*   **Production Phase 2:** $6840
*   **Post-Production:** $2640


In [15]:
import pandas as pd

# Reuse objects from Task 2
# prompts, pro_texts (baseline = Pro), flash_texts (candidate = Flash)

eval_df = pd.DataFrame({
    "prompt": prompts,                          # input to the task (optional for BYOR templates, but fine to keep)
    "baseline_model_response": pro_texts,       # baseline = Gemini Pro
    "response": flash_texts,                    # candidate = Gemini Flash
})

eval_df


,prompt,baseline_model_response,response
0,"<instructions>\n Prepare a document to fulfill the task based on the context provided.\n </instructions>\n<task>\n What is the cost of each phase of production? \n If days are mentioned, assume an 8 hour work day.\n </task>\n<context>\n Screenwriter: $40\nActor: $25\nDirector: $30\nCamera Operator: $35\nSound Engineer: $20\nEditor: $30\n\nPhases of Production:\n Writing:\n The Screenwriter will write the script.\n They need 72 hours to do so.\n\n\n Pre-Production:\n The Director needs time to analyze the script.\n They will work on it for 36 hours.\n The Camera Operator will join the director for 24 hours of planning.\n\n\n Production Phase 1\n The first three days of filming will require the director, 4 actors, the camera operator, and the sound engineer\n\n\n Production Phase 2\n The next three days of filming will require the director, 8 actors, the camera operator, and the sound engineer\n\n\n Post-Production\n The editor will take 64 hours to edit the film.\n The director will work with the editor for 24 hours during this phase.\n </context>","Based on the information provided, here is the cost breakdown for each phase of production.\n\n### **Writing**\n* **Screenwriter:** 72 hours @ $40/hour = $2,880\n* **Total Phase Cost: $2,880**\n\n---\n\n### **Pre-Production**\n* **Director:** 36 hours @ $30/hour = $1,080\n* **Camera Operator:** 24 hours @ $35/hour = $840\n* **Total Phase Cost: $1,920**\n\n---\n\n### **Production Phase 1**\nThis phase lasts for 3 days, which is a total of 24 work hours (3 days x 8 hours/day).\n* **Director:** 24 hours @ $30/hour = $720\n* **4 Actors:** 24 hours x 4 actors @ $25/hour = $2,400\n* **Camera Operator:** 24 hours @ $35/hour = $840\n* **Sound Engineer:** 24 hours @ $20/hour = $480\n* **Total Phase Cost: $4,440**\n\n---\n\n### **Production Phase 2**\nThis phase lasts for 3 days, which is a total of 24 work hours (3 days x 8 hours/day).\n* **Director:** 24 hours @ $30/hour = $720\n* **8 Actors:** 24 hours x 8 actors @ $25/hour = $4,800\n* **Camera Operator:** 24 hours @ $35/hour = $840\n* **Sound Engineer:** 24 hours @ $20/hour = $480\n* **Total Phase Cost: $6,840**\n\n---\n\n### **Post-Production**\n* **Editor:** 64 hours @ $30/hour = $1,920\n* **Director:** 24 hours @ $30/hour = $720\n* **Total Phase Cost: $2,640**",Here's a breakdown of the cost for each phase of production:\n\n**Writing:**\n\n* Screenwriter: 72 hours * $40/hour = $2880\n\n**Pre-Production:**\n\n* Director: 36 hours * $30/hour = $1080\n* Camera Operator: 24 hours * $35/hour = $840\n* **Total Pre-Production Cost:** $1080 + $840 = $1920\n\n**Production Phase 1 (3 days):**\n\n* Director: 3 days * 8 hours/day * $30/hour = $720\n* Actors (4): 3 days * 8 hours/day * $25/hour * 4 actors = $2400\n* Camera Operator: 3 days * 8 hours/day * $35/hour = $840\n* Sound Engineer: 3 days * 8 hours/day * $20/hour = $480\n* **Total Production Phase 1 Cost:** $720 + $2400 + $840 + $480 = $4440\n\n**Production Phase 2 (3 days):**\n\n* Director: 3 days * 8 hours/day * $30/hour = $720\n* Actors (8): 3 days * 8 hours/day * $25/hour * 8 actors = $4800\n* Camera Operator: 3 days * 8 hours/day * $35/hour = $840\n* Sound Engineer: 3 days * 8 hours/day * $20/hour = $480\n* **Total Production Phase 2 Cost:** $720 + $4800 + $840 + $480 = $6840\n\n**Post-Production:**\n\n* Editor: 64 hours * $30/hour = $1920\n* Director: 24 hours * $30/hour = $720\n* **Total Post-Production Cost:** $1920 + $720 = $2640\n\n**Summary of Costs:**\n\n* **Writing:** $2880\n* **Pre-Production:** $1920\n* **Production Phase 1:** $4440\n* **Production Phase 2:** $6840\n* **Post-Production:** $2640\n
1,"<instructions>\n Prepare a document to fulfill the task based on the context provided.\n </instructions>\n<task>\n How many days will each phase require? Assume an \n 8 hour work day. If multiple people are working in parallel, \n do not add those times together, but only use the longest time. \n Also include a count of the total number 

In [16]:
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

eval_task = EvalTask(
    dataset=eval_df,
    metrics=[MetricPromptTemplateExamples.Pairwise.QUESTION_ANSWERING_QUALITY],
    experiment="indie-film-planning",
)

eval_task


In [18]:
# ✅ BYOR evaluation: do NOT pass `model=` here
eval_result = eval_task.evaluate()


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 3 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 3/3 [00:04<00:00,  1.54s/it]
INFO:vertexai.evaluation._evaluation:All 3 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:4.656401110000161 seconds


In [19]:
from IPython.display import display

summary = getattr(eval_result, "summary_table", None)
metrics = getattr(eval_result, "metrics_table", None)

if summary is not None:
    display(summary)
else:
    print("No summary_table found.")

if metrics is not None:
    display(metrics)
else:
    print("No metrics_table found.")


No summary_table found.


,prompt,baseline_model_response,response,pairwise_question_answering_quality/explanation,pairwise_question_answering_quality/pairwise_choice
0,"<instructions>\n Prepare a document to fulfill the task based on the context provided.\n </instructions>\n<task>\n What is the cost of each phase of production? \n If days are mentioned, assume an 8 hour work day.\n </task>\n<context>\n Screenwriter: $40\nActor: $25\nDirector: $30\nCamera Operator: $35\nSound Engineer: $20\nEditor: $30\n\nPhases of Production:\n Writing:\n The Screenwriter will write the script.\n They need 72 hours to do so.\n\n\n Pre-Production:\n The Director needs time to analyze the script.\n They will work on it for 36 hours.\n The Camera Operator will join the director for 24 hours of planning.\n\n\n Production Phase 1\n The first three days of filming will require the director, 4 actors, the camera operator, and the sound engineer\n\n\n Production Phase 2\n The next three days of filming will require the director, 8 actors, the camera operator, and the sound engineer\n\n\n Post-Production\n The editor will take 64 hours to edit the film.\n The director will work with the editor for 24 hours during this phase.\n </context>","Based on the information provided, here is the cost breakdown for each phase of production.\n\n### **Writing**\n* **Screenwriter:** 72 hours @ $40/hour = $2,880\n* **Total Phase Cost: $2,880**\n\n---\n\n### **Pre-Production**\n* **Director:** 36 hours @ $30/hour = $1,080\n* **Camera Operator:** 24 hours @ $35/hour = $840\n* **Total Phase Cost: $1,920**\n\n---\n\n### **Production Phase 1**\nThis phase lasts for 3 days, which is a total of 24 work hours (3 days x 8 hours/day).\n* **Director:** 24 hours @ $30/hour = $720\n* **4 Actors:** 24 hours x 4 actors @ $25/hour = $2,400\n* **Camera Operator:** 24 hours @ $35/hour = $840\n* **Sound Engineer:** 24 hours @ $20/hour = $480\n* **Total Phase Cost: $4,440**\n\n---\n\n### **Production Phase 2**\nThis phase lasts for 3 days, which is a total of 24 work hours (3 days x 8 hours/day).\n* **Director:** 24 hours @ $30/hour = $720\n* **8 Actors:** 24 hours x 8 actors @ $25/hour = $4,800\n* **Camera Operator:** 24 hours @ $35/hour = $840\n* **Sound Engineer:** 24 hours @ $20/hour = $480\n* **Total Phase Cost: $6,840**\n\n---\n\n### **Post-Production**\n* **Editor:** 64 hours @ $30/hour = $1,920\n* **Director:** 24 hours @ $30/hour = $720\n* **Total Phase Cost: $2,640**",Here's a breakdown of the cost for each phase of production:\n\n**Writing:**\n\n* Screenwriter: 72 hours * $40/hour = $2880\n\n**Pre-Production:**\n\n* Director: 36 hours * $30/hour = $1080\n* Camera Operator: 24 hours * $35/hour = $840\n* **Total Pre-Production Cost:** $1080 + $840 = $1920\n\n**Production Phase 1 (3 days):**\n\n* Director: 3 days * 8 hours/day * $30/hour = $720\n* Actors (4): 3 days * 8 hours/day * $25/hour * 4 actors = $2400\n* Camera Operator: 3 days * 8 hours/day * $35/hour = $840\n* Sound Engineer: 3 days * 8 hours/day * $20/hour = $480\n* **Total Production Phase 1 Cost:** $720 + $2400 + $840 + $480 = $4440\n\n**Production Phase 2 (3 days):**\n\n* Director: 3 days * 8 hours/day * $30/hour = $720\n* Actors (8): 3 days * 8 hours/day * $25/hour * 8 actors = $4800\n* Camera Operator: 3 days * 8 hours/day * $35/hour = $840\n* Sound Engineer: 3 days * 8 hours/day * $20/hour = $480\n* **Total Production Phase 2 Cost:** $720 + $4800 + $840 + $480 = $6840\n\n**Post-Production:**\n\n* Editor: 64 hours * $30/hour = $1920\n* Director: 24 hours * $30/hour = $720\n* **Total Post-Production Cost:** $1920 + $720 = $2640\n\n**Summary of Costs:**\n\n* **Writing:** $2880\n* **Pre-Production:** $1920\n* **Production Phase 1:** $4440\n* **Production Phase 2:** $6840\n* **Post-Production:** $2640\n,BASELINE response is slightly better formatted for readability. Both responses were very similar in quality and did a great job answering the question.,BASELINE
1,"<instructions>\n Prepare a document to fulfill the task based on the context p

In [20]:
# Preferred response per row
if metrics is not None and "pairwise_choice" in metrics.columns:
    display(metrics[["pairwise_choice"]])

# Explanations per row
if metrics is not None:
    exp_col = "explanation" if "explanation" in metrics.columns else None
    if exp_col:
        display(metrics[[exp_col]])


In [21]:
import pandas as pd

if metrics is not None and "pairwise_choice" in metrics.columns:
    counts = metrics["pairwise_choice"].value_counts(dropna=False)
    baseline_wins  = int(counts.get("baseline", 0))   # Gemini 2.5 Pro
    candidate_wins = int(counts.get("candidate", 0))  # Gemini 2.0 Flash
    ties_other     = int(counts.sum() - baseline_wins - candidate_wins)

    display(pd.DataFrame({
        "Model": ["Gemini 2.5 Pro (baseline)", "Gemini 2.0 Flash (candidate)", "Ties/Other"],
        "Wins":  [baseline_wins, candidate_wins, ties_other]
    }))

    winner = ("Gemini 2.5 Pro" if baseline_wins > candidate_wins
              else "Gemini 2.0 Flash" if candidate_wins > baseline_wins
              else "Tie")
    print(f"Preferred model based on pairwise choices: {winner}")
